# Catalan Accent Oracle: CV26 Cloud Experiment

Run the Common Voice 26 Catalan HuBERT embedding pipeline in Google Colab or Kaggle.

This notebook mirrors the repository scripts and defaults to a quota-friendly smoke run: 25 speakers per dialect and 2 clips per speaker. The current best local setup is `BSC-LT/hubert-base-ca-2k` HuBERT embeddings plus a calibrated linear SVM.

**Recommended:** prepare a small upload bundle on your PC (see the next section) instead of uploading the full ~85 GB CV26 archive.

Expected inputs:
- A clone or uploaded copy of this repository.
- A small bundle zip from `scripts/package_cloud_upload_bundle.py`, **or** the full archive `common-voice-scripted-speech-26-0-catala-fe69b989.tar.gz` if you can mount it.

No secrets or tokens are required.


## 0. Local Prep → Upload To Drive / Kaggle

Do **not** upload the full CV26 archive (`common-voice-scripted-speech-26-0-catala-fe69b989.tar.gz`, ~85 GB). Colab and Kaggle upload limits make that impractical.

Instead, on your PC (with the archive at `data/raw/`), build a small bundle and upload only that zip plus this repository.

### Option A — Audio bundle (cloud runs HuBERT)

Extracts only the manifest clips locally. Upload size is larger, but the cloud runtime still computes embeddings (useful if you want GPU embedding extraction in Colab/Kaggle).

```bash
cd /path/to/proj-accents
source .venv/bin/activate

# Smoke (~250 clips, ~8 MB zip)
python scripts/package_cloud_upload_bundle.py --mode smoke --bundle-type audio

# Full 1,440-clip run (~46 MB zip) — reuses existing local artifacts when available
python scripts/package_cloud_upload_bundle.py --mode full_1440 --bundle-type audio --skip-prep
```

### Option B — Embeddings bundle (recommended, smallest/fastest cloud run)

Pre-computes HuBERT embeddings locally. Cloud notebook skips audio extraction and embedding extraction; it goes straight to baseline training and model export.

```bash
cd /path/to/proj-accents
source .venv/bin/activate

# Smoke (~250 clips, ~2 MB zip). Local embedding step uses --device cuda if you have a GPU.
python scripts/package_cloud_upload_bundle.py --mode smoke --bundle-type embeddings --device cuda

# Full 1,440-clip run (~9 MB zip) — reuses existing local artifacts when available
python scripts/package_cloud_upload_bundle.py --mode full_1440 --bundle-type embeddings --skip-prep
```

### Upload checklist

| Upload | Skip |
| --- | --- |
| Repository source (or open notebook from GitHub) | Full `common-voice-scripted-speech-26-0-catala-fe69b989.tar.gz` |
| `data/cloud_bundles/smoke-embeddings.zip` or `full_1440-embeddings.zip` | Entire `data/metadata/cv26-ca/` folder (only needed locally) |
| Same paths for `*-audio.zip` if using Option A | |

Typical bundle output paths:
- `data/cloud_bundles/smoke-embeddings.zip`
- `data/cloud_bundles/smoke-audio.zip`
- `data/cloud_bundles/full_1440-embeddings.zip`
- `data/cloud_bundles/full_1440-audio.zip`

In the next section set `INPUT_SOURCE = "embeddings_bundle"` (or `"audio_bundle"`) and point `BUNDLE_CANDIDATES` at your uploaded zip.

## 1. Install Dependencies

Colab and Kaggle usually include many scientific packages, but the exact versions vary. Run this cell once per fresh runtime. Restart the kernel only if the notebook asks you to after package installation.


In [ ]:
%pip install -q -r requirements.txt


## 2. Configure Environment And Paths

Set `RUN_MODE = "smoke"` for a quick end-to-end check. Change it to `"full_1440"` when you are ready to reproduce the 1,440-clip experiment described in `reports/cv26_train_1440_experiment.md`.

Set `INPUT_SOURCE` to match what you uploaded:
- `"embeddings_bundle"` — recommended; smallest upload, skips HuBERT extraction in the cloud
- `"audio_bundle"` — upload MP3s only; cloud still runs HuBERT
- `"archive"` — only if the full tar.gz is mounted locally in the runtime

For Colab, you can optionally mount Google Drive by setting `MOUNT_GOOGLE_DRIVE = True`. Kaggle users usually place uploaded files under `/kaggle/input` and write outputs under `/kaggle/working`.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
import torch

RUN_MODE = "smoke"  # "smoke" or "full_1440"
INPUT_SOURCE = "embeddings_bundle"  # "embeddings_bundle", "audio_bundle", or "archive"
MOUNT_GOOGLE_DRIVE = False

if RUN_MODE == "smoke":
    RUN_TAG = "cv26-cloud-smoke"
    MAX_SPEAKERS_PER_LABEL = 25
    MAX_CLIPS_PER_SPEAKER = 2
    EMBEDDING_MAX_ROWS = None
elif RUN_MODE == "full_1440":
    RUN_TAG = "cv26-train-1440-cloud"
    MAX_SPEAKERS_PER_LABEL = 150
    MAX_CLIPS_PER_SPEAKER = 3
    EMBEDDING_MAX_ROWS = None
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'full_1440'")

REPO_DIR = Path.cwd()
ARCHIVE_NAME = "common-voice-scripted-speech-26-0-catala-fe69b989.tar.gz"
OUTPUT_ROOT = Path("/kaggle/working/proj-accents") if Path("/kaggle/working").exists() else REPO_DIR
METADATA_DIR = REPO_DIR / "data/metadata/cv26-ca"
BUNDLE_ROOT = OUTPUT_ROOT

ARCHIVE_CANDIDATES = [
    REPO_DIR / "data/raw" / ARCHIVE_NAME,
    Path("/content/drive/MyDrive/proj-accents") / ARCHIVE_NAME,
    Path("/content") / ARCHIVE_NAME,
    Path("/kaggle/input") / ARCHIVE_NAME,
]

BUNDLE_SUFFIX = "embeddings" if INPUT_SOURCE == "embeddings_bundle" else "audio"
BUNDLE_CANDIDATES = [
    REPO_DIR / "data/cloud_bundles" / f"{RUN_MODE}-{BUNDLE_SUFFIX}.zip",
    OUTPUT_ROOT / "data/cloud_bundles" / f"{RUN_MODE}-{BUNDLE_SUFFIX}.zip",
    Path("/content/drive/MyDrive/proj-accents") / f"{RUN_MODE}-{BUNDLE_SUFFIX}.zip",
    Path("/content") / f"{RUN_MODE}-{BUNDLE_SUFFIX}.zip",
    Path("/kaggle/input") / f"{RUN_MODE}-{BUNDLE_SUFFIX}.zip",
]

MANIFEST_PATH = BUNDLE_ROOT / "manifests" / f"{RUN_TAG}.csv"
AUDIO_DIR = BUNDLE_ROOT / "data/audio" / RUN_TAG
EMBEDDINGS_DIR = BUNDLE_ROOT / "embeddings" / RUN_TAG
BASELINE_DIR = OUTPUT_ROOT / "reports/baselines" / RUN_TAG
MODEL_DIR = OUTPUT_ROOT / "models" / f"{RUN_TAG}-hubert-svm-calibrated"
prepared_manifest = AUDIO_DIR / "prepared_manifest.csv"
embedding_index = EMBEDDINGS_DIR / "embedding_index.csv"
bundle_info: dict | None = None

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped: {type(exc).__name__}: {exc}")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Repo:", REPO_DIR)
print("Output root:", OUTPUT_ROOT)
print("Bundle root:", BUNDLE_ROOT)
print("CUDA available:", torch.cuda.is_available())
print("Input source:", INPUT_SOURCE)


## 3. Load Bundle Or Locate Archive

If you uploaded a bundle zip, the next cell extracts it under `BUNDLE_ROOT` and sets manifest/audio/embedding paths from `bundle_manifest.json`.

If `INPUT_SOURCE = "archive"`, the notebook instead looks for the CV26 tar.gz and ensures `data/metadata/cv26-ca/train.tsv` exists (extracting only TSV/README files from the archive when needed).


In [ ]:
def find_first(candidates: list[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def resolve_bundle_paths(index_path: Path, root: Path) -> Path:
    df = pd.read_csv(index_path)
    for column in ("audio_path", "embedding_path"):
        if column in df.columns:
            df[column] = df[column].map(
                lambda value: str(root / value) if value and not Path(value).is_absolute() else value
            )
    resolved = index_path.with_name(index_path.stem + ".resolved.csv")
    df.to_csv(resolved, index=False)
    return resolved

def resolve_prepared_manifest(path: Path, root: Path) -> Path:
    df = pd.read_csv(path)
    if "audio_path" in df.columns:
        df["audio_path"] = df["audio_path"].map(
            lambda value: str(root / value) if value and not Path(value).is_absolute() else value
        )
    resolved = path.with_name(path.stem + ".resolved.csv")
    df.to_csv(resolved, index=False)
    return resolved

required_scripts = [
    "scripts/build_cv26_balanced_manifest.py",
    "scripts/prepare_cv26_audio_from_archive.py",
    "scripts/extract_hubert_embeddings.py",
    "scripts/train_embedding_baselines.py",
    "scripts/train_embedding_model_artifact.py",
]
missing_scripts = [path for path in required_scripts if not (REPO_DIR / path).exists()]
if missing_scripts:
    raise FileNotFoundError(f"Missing expected repository scripts: {missing_scripts}")

archive_path = find_first(ARCHIVE_CANDIDATES) if INPUT_SOURCE == "archive" else None
bundle_zip = find_first(BUNDLE_CANDIDATES) if INPUT_SOURCE in {"audio_bundle", "embeddings_bundle"} else None

if INPUT_SOURCE in {"audio_bundle", "embeddings_bundle"}:
    if bundle_zip is None:
        raise FileNotFoundError(
            f"Upload the {RUN_MODE}-{BUNDLE_SUFFIX}.zip bundle and add its path to BUNDLE_CANDIDATES."
        )
    marker = BUNDLE_ROOT / "bundle_manifest.json"
    if not marker.exists():
        with zipfile.ZipFile(bundle_zip, "r") as zf:
            zf.extractall(BUNDLE_ROOT)
    bundle_info = json.loads(marker.read_text(encoding="utf-8"))
    if bundle_info["mode"] != RUN_MODE:
        raise ValueError(f"Bundle mode {bundle_info['mode']!r} does not match RUN_MODE={RUN_MODE!r}.")
    if bundle_info["bundle_type"] != BUNDLE_SUFFIX:
        raise ValueError(
            f"Bundle type {bundle_info['bundle_type']!r} does not match INPUT_SOURCE={INPUT_SOURCE!r}."
        )

    MANIFEST_PATH = BUNDLE_ROOT / bundle_info["manifest_path"]
    AUDIO_DIR = BUNDLE_ROOT / bundle_info["audio_dir"]
    EMBEDDINGS_DIR = BUNDLE_ROOT / bundle_info["embeddings_dir"]
    prepared_manifest = BUNDLE_ROOT / bundle_info["prepared_manifest_path"]
    if bundle_info.get("embedding_index_path"):
        embedding_index = resolve_bundle_paths(
            BUNDLE_ROOT / bundle_info["embedding_index_path"],
            BUNDLE_ROOT,
        )
    else:
        embedding_index = EMBEDDINGS_DIR / "embedding_index.csv"

    print("Bundle zip:", bundle_zip)
    print(json.dumps(bundle_info, indent=2))
elif INPUT_SOURCE == "archive":
    print("Archive:", archive_path if archive_path else "not found")
    if not archive_path and not (METADATA_DIR / "train.tsv").exists():
        raise FileNotFoundError(
            "Upload or mount the CV26 Catalan tar.gz archive, or provide data/metadata/cv26-ca/train.tsv."
        )
else:
    raise ValueError("INPUT_SOURCE must be 'embeddings_bundle', 'audio_bundle', or 'archive'")


In [ ]:
if INPUT_SOURCE == "archive":
    def extract_cv26_metadata_only(archive: Path, metadata_dir: Path) -> None:
        metadata_dir.mkdir(parents=True, exist_ok=True)
        wanted_suffixes = (".tsv", ".txt", ".md")
        extracted = []
        with tarfile.open(archive, "r:gz") as tar:
            for member in tar:
                name = member.name
                if "/ca/" not in name or not name.lower().endswith(wanted_suffixes):
                    continue
                if "/clips/" in name:
                    continue
                source = tar.extractfile(member) if member.isfile() else None
                if source is None:
                    continue
                target = metadata_dir / Path(name).name
                with target.open("wb") as fh:
                    shutil.copyfileobj(source, fh)
                extracted.append(target.name)
        print(f"Extracted metadata files: {sorted(set(extracted))}")

    if not (METADATA_DIR / "train.tsv").exists():
        extract_cv26_metadata_only(archive_path, METADATA_DIR)

    for split in ["train", "dev", "test"]:
        path = METADATA_DIR / f"{split}.tsv"
        print(split, "exists=" + str(path.exists()), "size=" + (str(path.stat().st_size) if path.exists() else "n/a"))

    if not (METADATA_DIR / "train.tsv").exists():
        raise FileNotFoundError("train.tsv is still missing after metadata extraction.")
else:
    print("Skipping metadata extraction; using pre-built bundle contents.")


## 4. Confirm Experiment Paths

`RUN_MODE` is set in section 2. For bundle uploads, manifest and prepared audio paths were already loaded from `bundle_manifest.json`. Archive mode will build these in the next sections.


In [ ]:
print(json.dumps({
    "run_mode": RUN_MODE,
    "input_source": INPUT_SOURCE,
    "manifest": str(MANIFEST_PATH),
    "audio_dir": str(AUDIO_DIR),
    "prepared_manifest": str(prepared_manifest),
    "embeddings_dir": str(EMBEDDINGS_DIR),
    "embedding_index": str(embedding_index),
    "baseline_dir": str(BASELINE_DIR),
    "model_dir": str(MODEL_DIR),
}, indent=2))

if INPUT_SOURCE != "archive" and MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
    print("Rows:", len(manifest))
    print("Speakers:", manifest["client_id"].nunique())
    display(manifest.groupby("label").agg(rows=("path", "count"), speakers=("client_id", "nunique")))


## 5. Build A Balanced Manifest

Archive mode only. Bundle uploads already include the manifest built locally with the same `RUN_MODE`, seed, and label policy.


In [ ]:
def run_cmd(args: list[str]) -> None:
    print("$", " ".join(args))
    subprocess.run(args, cwd=REPO_DIR, check=True)

if INPUT_SOURCE == "archive":
    run_cmd([
        sys.executable,
        "scripts/build_cv26_balanced_manifest.py",
        "--metadata-dir", str(METADATA_DIR),
        "--source-split", "train",
        "--out-manifest", str(MANIFEST_PATH),
        "--max-speakers-per-label", str(MAX_SPEAKERS_PER_LABEL),
        "--max-clips-per-speaker", str(MAX_CLIPS_PER_SPEAKER),
        "--seed", "13",
    ])

manifest = pd.read_csv(MANIFEST_PATH)
print("Rows:", len(manifest))
print("Speakers:", manifest["client_id"].nunique())
display(manifest.groupby("label").agg(rows=("path", "count"), speakers=("client_id", "nunique")))
manifest.head()


## 6. Extract Only Selected Audio

Archive mode only. Bundle uploads already include the selected MP3 files and `prepared_manifest.csv`.


In [ ]:
if INPUT_SOURCE == "archive":
    if archive_path is None:
        raise FileNotFoundError("Audio extraction needs the CV26 tar.gz archive.")

    run_cmd([
        sys.executable,
        "scripts/prepare_cv26_audio_from_archive.py",
        "--archive", str(archive_path),
        "--manifest", str(MANIFEST_PATH),
        "--out-dir", str(AUDIO_DIR),
    ])

prepared = pd.read_csv(prepared_manifest)
print("Prepared rows:", int(prepared["audio_prepared"].sum()), "/", len(prepared))
prepared.head()


## 7. Extract HuBERT Embeddings

Skip this section when `INPUT_SOURCE = "embeddings_bundle"`.

For archive or audio bundles, this is the most expensive step. It uses GPU when available and falls back to CPU otherwise.


In [ ]:
if INPUT_SOURCE in {"archive", "audio_bundle"}:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    manifest_for_embeddings = resolve_prepared_manifest(prepared_manifest, BUNDLE_ROOT)
    embedding_cmd = [
        sys.executable,
        "scripts/extract_hubert_embeddings.py",
        "--prepared-manifest", str(manifest_for_embeddings),
        "--out-dir", str(EMBEDDINGS_DIR),
        "--model-name", "BSC-LT/hubert-base-ca-2k",
        "--device", device,
        "--force-exit",
    ]
    if EMBEDDING_MAX_ROWS is not None:
        embedding_cmd.extend(["--max-rows", str(EMBEDDING_MAX_ROWS)])

    run_cmd(embedding_cmd)
    embedding_index = resolve_bundle_paths(EMBEDDINGS_DIR / "embedding_index.csv", BUNDLE_ROOT)
else:
    print("Skipping embedding extraction; using pre-computed bundle embeddings.")

embeddings = pd.read_csv(embedding_index)
print("Embedded rows:", len(embeddings))
print("Embedding dim:", embeddings["embedding_dim"].dropna().iloc[0] if len(embeddings) else "n/a")
embeddings.head()


## 8. Train And Evaluate Baselines

The baseline script uses speaker-grouped cross-validation. For very small smoke runs it automatically reduces the fold count when needed.


In [ ]:
run_cmd([
    sys.executable,
    "scripts/train_embedding_baselines.py",
    "--embedding-index", str(embedding_index),
    "--out-dir", str(BASELINE_DIR),
])

results_path = BASELINE_DIR / "results.json"
results = json.loads(results_path.read_text(encoding="utf-8"))
pd.DataFrame(results["results"])[["model", "accuracy", "macro_f1", "top2_accuracy"]]


## 9. Save The Calibrated SVM Model Artifact

This trains the current best artifact shape on all embeddings from this run: StandardScaler plus calibrated linear SVM. The saved artifact can be copied back into the repository or used by the local inference API after matching paths and metadata.


In [ ]:
run_cmd([
    sys.executable,
    "scripts/train_embedding_model_artifact.py",
    "--train-index", str(embedding_index),
    "--out-dir", str(MODEL_DIR),
    "--encoder-model-name", "BSC-LT/hubert-base-ca-2k",
])

print("Model files:")
for path in sorted(MODEL_DIR.glob("*")):
    print("-", path, path.stat().st_size, "bytes")
json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))


## 10. Package Key Artifacts

Run this when you want a compact download containing the manifest, summaries, baseline reports, model artifact, and embedding index. Embedding vector files and audio clips can be large, so they are excluded by default.


In [ ]:
ARTIFACT_ZIP = OUTPUT_ROOT / f"{RUN_TAG}-key-artifacts.zip"
paths_to_zip = [
    MANIFEST_PATH,
    MANIFEST_PATH.with_suffix(".summary.md"),
    AUDIO_DIR / "summary.json",
    EMBEDDINGS_DIR / "summary.json",
    EMBEDDINGS_DIR / "embedding_index.csv",
    BASELINE_DIR / "results.json",
    BASELINE_DIR / "results.md",
    MODEL_DIR / "model.joblib",
    MODEL_DIR / "metadata.json",
]

import zipfile
with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in paths_to_zip:
        if path.exists():
            zf.write(path, arcname=str(path.relative_to(OUTPUT_ROOT) if path.is_relative_to(OUTPUT_ROOT) else path.name))

print("Wrote", ARTIFACT_ZIP, ARTIFACT_ZIP.stat().st_size, "bytes")

try:
    from google.colab import files
    # Uncomment to download automatically in Colab.
    # files.download(str(ARTIFACT_ZIP))
except Exception:
    pass


## Notes For Scaling Up

- Prepare bundles locally with `scripts/package_cloud_upload_bundle.py`; upload the zip, not the ~85 GB archive.
- Prefer `INPUT_SOURCE = "embeddings_bundle"` for the smallest upload and fastest cloud run.
- Keep `RUN_MODE = "smoke"` until every step completes once in your runtime.
- Switch to `RUN_MODE = "full_1440"` to reproduce the strongest local training subset.
- Prefer GPU for embedding extraction when using `audio_bundle` or `archive` mode.
- Do not combine CV26 official train/dev/test splits as independent data unless you are intentionally designing a new split policy.
- Keep held-out or manually recorded evaluation separate from the training manifest to avoid speaker leakage.
